# ProChem core functionality demo

Этот notebook показывает текущий уровень проработки ядра `prochem`: модели данных, VASP I/O, склейку restart-расчетов, dataset-режим для MLIP, анализ последовательностей структур и экспорт таблиц.

Все пути ниже можно менять под локальные расчеты. Notebook рассчитан на запуск из корня репозитория `B:\\Science\\prochem`.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

ROOT, SRC

In [ ]:
import numpy as np
import pandas as pd

from prochem.analysis import (
    AnalysisTable,
    Selection,
    export_dataframe,
)
from prochem.adapters.jupyter import scene_animation, scene_figure
from prochem.core import (
    Atom,
    BandStructure,
    Calculation,
    Cell,
    DensityOfStates,
    Structure,
    Structures,
    StructuresMergePolicy,
)
from prochem.io import parse
from prochem.io.vasp import (
    discover_structure_files,
    discover_vasprun_files,
    parse_structure_dataset,
    parse_structure_dataset_as_structures,
    parse_vasprun_sequence,
)
from prochem.rendering import to_scene_data


## Базовая модель: Atom -> Structure -> Structures

`Atom` содержит данные элемента и состояние конкретного атома. `name` можно задавать символом (`"C"`) или порядковым номером элемента (`6`). Потенциальная, кинетическая и полная энергии хранятся отдельно: `potential_energy`, `kinetic_energy`, `total_energy`; если энергия неизвестна, значение равно `None`, а в плотных массивах используется `NaN`. `Structure` хранит список `Atom` и скалярные энергии одного ионного шага, а `Structures` хранит последовательность кадров, `timestep` и `sources`. Per-atom kinetic energy можно вычислить из velocity, но per-atom potential/total energy не размазывается из энергии структуры автоматически. Визуальные поля `size` и `color` общие для всех атомов с одинаковым `name`, поэтому стиль элемента можно менять один раз.


In [ ]:
carbon_a = Atom(name="C")
carbon_b = Atom(name=6)

carbon_a.size = 1.2
carbon_a.color = (0.1, 0.1, 0.1, 1.0)

example_structure = Structure(
    atoms=[
        Atom(name="C", index=0, position=(0.0, 0.0, 0.0), velocity=(1.0, 0.0, 0.0)),
        Atom(name="O", index=1, position=(1.2, 0.0, 0.0), potential_energy=0.05),
    ],
    cell=Cell(np.eye(3) * 10.0),
    potential_energy=-10.5,
)
example_structures = Structures(frames=[example_structure], timestep=1.0, sources=[ROOT])

print(carbon_a.mass, carbon_a.charge, carbon_a.valent_charge)
print(carbon_a.size is carbon_b.size, carbon_a.color is carbon_b.color)
print(example_structure.atoms[0].kinetic_energy_from_velocity())
print(example_structure.atoms[0].approximate_force((2.0, 0.0, 0.0), delta_t=0.5))
print(example_structure.positions)
print(example_structure.potential_energy, example_structure.atom_potential_energies_array())
print(example_structures.time_fs)


## Тестовые пути

`MAIN_VASPRUN` - одиночный `vasprun.xml`.
`DELETION_RESTART_DIR` - пример restart, где при продолжении удалялся атом.
`EXACT_RESTART_DIR` - пример restart с обычными exact overlap-сегментами.

In [ ]:
MAIN_VASPRUN = Path(r"/mnt/b/Science/Calculations/VASP/Low-k/Poss_with_Ar/POSS_Ar_30_grad_20eV/vasprun.xml")
EXAMPLE_OUTCAR = Path(r"/mnt/b/Science/Calculations/VASP/ALE/C10F22/Ar/C-C/20eV/OUTCAR")
DELETION_RESTART_DIR = Path(r"/mnt/b/Science/Calculations/VASP/ALE/C12F26/Ar/C/30eV")
EXACT_RESTART_DIR = Path(r"/mnt/b/Science/Calculations/VASP/MoS2/N2/Mo/MoS2_N2_30eV_parallel_Mo")

for path in [MAIN_VASPRUN, EXAMPLE_OUTCAR, DELETION_RESTART_DIR, EXACT_RESTART_DIR]:
    print(path, "exists=", path.exists())

## Парсинг: Calculation и Structures

`parse()` выбирает парсер через `prochem.io.registry`. Для VASP возвращается единая модель `Calculation`.

In [ ]:
calculation = parse(MAIN_VASPRUN)
structures = calculation.structures
first = structures.frame(0)

print(type(calculation).__name__, calculation.engine, calculation.source)
print("steps:", calculation.step_count)
print("registry atoms:", calculation.atom_count)
print("timestep:", structures.timestep)
print("first frame atoms:", first.atom_count)
print("cell:\n", first.cell.vectors)
print("first species:", first.species[:10])
print("all species:", first.species)

In [ ]:
structures.frame(1)

## Плотные массивы Structures

`Structures` хранит список `Structure`, но умеет отдавать плотные массивы координат, скоростей, сил и энергий. Если атом был удален при restart, его слоты заполняются `NaN`.

In [ ]:
positions = structures.positions_array()
direct = structures.direct_positions_array()
velocities_dense = structures.velocities_array()
forces_dense = structures.forces_array()
atom_potential_energies = structures.atom_potential_energies_array()
atom_kinetic_energies = structures.atom_kinetic_energies_array()
structure_potential_energies = structures.structure_potential_energies_array()
structure_kinetic_energies = structures.structure_kinetic_energies_array()
mask = structures.presence_mask()

print("positions:", positions.shape)
print("direct:", direct.shape)
print("velocities:", None if velocities_dense is None else velocities_dense.shape)
print("forces:", None if forces_dense is None else forces_dense.shape)
print("atom potential energies:", atom_potential_energies.shape)
print("atom kinetic energies:", atom_kinetic_energies.shape)
print("structure potential energies:", structure_potential_energies.shape)
print("structure kinetic energies:", structure_kinetic_energies.shape)
print("sources:", len(structures.sources))
print("presence mask:", mask.shape)
print("missing atom slots:", int((~mask).sum()))

## Склейка restart-расчетов

`StructuresMergePolicy.boundary_search_frames` ищет overlap не только в первом кадре следующего `vasprun`, но и в первых N кадрах. Это нужно для случая, когда POSCAR был сделан из CONTCAR с оставленным блоком после скоростей: VASP может дать один или несколько стартовых кадров, которые не совпадают с концом предыдущего сегмента.

`allow_mismatch_fallback=False` запрещает fallback, при котором несовпадающий сегмент регистрируется как новые атомы. Это полезно для строгой проверки restart-склейки и для MLIP dataset workflow.

In [ ]:
policy = StructuresMergePolicy(
    boundary_search_frames=6,
    allow_mismatch_fallback=False,
    keep_topology_change_frame=True,
)

merged, report = parse_vasprun_sequence(DELETION_RESTART_DIR, policy=policy)
for event in report.events:
    print(
        event.status,
        "next=", event.next_source.name,
        "matched=", event.matched_next_frame,
        "drop=", event.dropped_next_frames,
        "deleted=", event.deleted_atom_ids,
        "max_delta=", event.max_delta,
    )

print("merged steps:", merged.step_count)
print("registry atoms:", merged.atom_count)
print("missing slots:", int((~merged.structures.presence_mask()).sum()))

## Analysis: координаты, скорости, энергии, расстояния

Функции из `prochem.analysis` не зависят от GUI. Их можно использовать из Qt, Jupyter и web API.

In [ ]:
structures_for_analysis = merged.structures
atom_ids = list(structures_for_analysis.atom_ids[:3])
selection = Selection("first3", atom_ids)

analysis = AnalysisTable(structures_for_analysis, selection)
analysis.add_coordinates("first3")
analysis.add_atom_velocities("first3")
analysis.add_atom_kinetic_energies("first3")
analysis.add_distances("first3", pairs=[(atom_ids[0], atom_ids[1])])
df = analysis.dataframe()
df.head()


In [ ]:
analysis.add_center_of_mass("first3")
analysis.add_center_of_mass_velocity("first3")
analysis.add_center_of_mass_kinetic_energy("first3")
analysis.data[["cm_first3_x", "cm_first3_y", "cm_first3_z", "V_cm_first3", "E_cm_first3"]].head()


## SceneData: единый DTO для визуализации

`to_scene_data()` превращает `Structure`, `Structures`, `StructureDataset` или `Calculation` в backend-independent `SceneData`. Qt/OpenGL, Jupyter/Plotly и web-слой могут читать одни и те же primitives: atoms, bonds, cell, axes. Для VESTA-like отображения periodic images ограничиваются `periodic_image_depth` и расстоянием до ячейки через `periodic_image_cutoff` или `periodic_image_cutoff_fraction`.


In [ ]:
scene = to_scene_data(
    merged,
    frame_indices=[0, merged.step_count // 2, merged.step_count - 1],
    atom_colors={"Si": "#d9b36c", "O": "#e74c3c", "H": "#ffffff", "C": "#777777", "F": "964B00"},
    atom_radius_scales={"H": 0.7, "Si": 1.15, "C": 2.3, "F": 1.7},
    bond_max_lengths={"Si-O": 2.1, "O-H": 1.2, "C-F": 1.9, "C-C": 2.1},
    include_periodic_images=True,
    periodic_image_depth=1,
    periodic_image_cutoff=4.0,
)

print(type(scene).__name__, scene.metadata)
print("frames:", scene.frame_count)
for frame in scene.frames:
    print(
        "frame", frame.metadata.get("frame_index"),
        "atoms", len(frame.atoms),
        "image_atoms", frame.metadata.get("image_atom_count"),
        "bonds", len(frame.bonds),
        "cell", frame.cell is not None,
    )


## Web schemas: SceneData JSON contract

`SceneDataSchema` задает JSON-контракт для будущего web API и может валидировать DTO без запуска FastAPI.

In [ ]:
try:
    from prochem.adapters.web.schemas import SceneDataSchema

    scene_payload = SceneDataSchema.from_scene_data(scene).model_dump(mode="json")
    {
        "name": scene_payload["name"],
        "frame_count": scene_payload["frame_count"],
        "first_frame_atoms": len(scene_payload["frames"][0]["atoms"]),
        "metadata": scene_payload["metadata"],
    }
except RuntimeError as exc:
    print(exc)


## Minimal Jupyter/Plotly adapter

`scene_figure()` строит Plotly 3D для одного frame, а `scene_animation()` добавляет slider по кадрам. Эти функции работают поверх `SceneData`, поэтому не зависят от VASP/Quantum ESPRESSO/LAMMPS напрямую.


In [ ]:
try:
    fig = scene_figure(scene, frame_index=0, title="First and last merged frames")
    fig.show()
except RuntimeError as error:
    print(error)
    print("Install with: pip install -e .[jupyter]")

In [ ]:
try:
    fig = scene_animation(scene, title="First and last merged frames")
    fig.update_layout(width=1920, height=1080)
    fig.show()
except RuntimeError as error:
    print(error)
    print("Install with: pip install -e .[jupyter]")


## Typed VASP result models

`OSZICAR`, `DOSCAR` и `EIGENVAL` теперь не раскладываются в набор разрозненных `dict`-полей. Парсер возвращает typed-модели через удобные свойства `Calculation`: `electronic_steps`, `ionic_steps`, `density_of_states`, `band_structure`. У каждой модели есть `to_dataframe()` для notebook/GUI/export сценариев.


In [ ]:
vasp_result_files = {
    "OSZICAR": DELETION_RESTART_DIR / "OSZICAR",
    "DOSCAR": MAIN_VASPRUN.with_name("DOSCAR"),
    "EIGENVAL": MAIN_VASPRUN.with_name("EIGENVAL"),
}

for label, path in vasp_result_files.items():
    print(f"\n{label}: {path}")
    if not path.exists():
        print("file not found in the demo calculation")
        continue

    result = parse(path)
    if result.errors.exist:
        print(result.errors.message)
        continue

    if result.ionic_steps is not None:
        display(result.ionic_steps.to_dataframe().head())
    if result.electronic_steps is not None:
        display(result.electronic_steps.to_dataframe().head())
    if result.density_of_states is not None:
        dos = result.density_of_states
        print(type(dos).__name__, "nedos=", dos.nedos, "efermi=", dos.fermi_energy)
        display(dos.to_dataframe(shifted=True).head())
    if result.band_structure is not None:
        bands = result.band_structure
        print(type(bands).__name__, "kpoints=", bands.kpoint_count, "bands=", bands.band_count)
        display(bands.to_dataframe().head())


## MLIP dataset mode

Для датасетов структуры считаются независимыми конфигурациями, а не restart-сегментами. Поэтому используется `discover_structure_files()` / `parse_structure_dataset()` / `parse_structure_dataset_as_structures()`. Потенциальная и кинетическая энергии датасета хранятся в каждом `Structure`, а не отдельными массивами на уровне `StructureDataset`.

По умолчанию на каждую подпапку выбирается `CONTCAR`, а если его нет - `POSCAR`. При упаковке в одну Structures включен `strict_topology=True`: если атомный состав или порядок видов отличается, функция падает вместо fallback-регистрации новых атомов.

In [ ]:
structure_files = discover_structure_files(DELETION_RESTART_DIR, recursive=True)
print("found structure files:", len(structure_files))
for path in structure_files[:10]:
    print(path)

In [ ]:
# Пример упаковки dataset в structures. Раскомментируйте на директории,
# где все структуры имеют одинаковую топологию и порядок атомов.
# dataset_calc = parse_structure_dataset_as_structures(DELETION_RESTART_DIR, recursive=True)
# print(dataset_calc.step_count, dataset_calc.atom_count)
# dataset_calc.structures.frame(0).properties

## Export

`export_dataframe()` умеет писать `.csv`, `.xlsx`, `.html`. Ниже пример CSV в локальную папку `notebooks/output`.

In [ ]:
output_dir = ROOT / "notebooks" / "output"
output_dir.mkdir(exist_ok=True)
output_path = export_dataframe(df.head(100), output_dir / "core_demo_table.csv")
output_path

## Что уже реализовано и что проверить дальше

- Готово: базовые `Atom/Structure/Structures/Calculation`, отдельный `StructureDataset`, потенциальная и кинетическая энергии на уровне `Atom` и `Structure`, VASP parser registry, restart-склейка, удаленные атомы после restart, плотные массивы с `NaN`, табличный анализ и экспорт.
- Готово по VASP-result model: `ElectronicStep/ElectronicSteps`, `IonicStep/IonicSteps`, `DensityOfStates`, `ProjectedDensityOfStates`, `BandStructure`; доступ через свойства `Calculation` и `to_dataframe()` для notebook/GUI.
- Готово по SceneData: `to_scene_data()` и специализированные конвертеры для `Structure`, `Structures`, `StructureDataset`, `Calculation`; primitives включают atoms, inferred bonds, cell и axes.
- Готово по Jupyter: `scene_figure()`, `scene_animation()`, `structures_slider()` поверх `SceneData`.
- Следующий шаг: расширить pytest-набор на restart-boundary случаи с несколькими synthetic `vasprun.xml` и добавить визуальную проверку Plotly в окружении с `prochem[jupyter]`.
